# Databases Case



🔨 Run the code cell below first to import the required packages.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Required for intermediate checks and autograder scripts
import unittest
import glob
import os
tc = unittest.TestCase()

🔨 Run the code cell below to create a helper function that deletes all rows from a database table. This function is used throughout the case to help you.

In [3]:
def delete_all_rows_from_table(table_to_truncate):
    conn_check = sqlite3.connect('NWT.db')

    tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_check)['tbl_name'])
    if table_to_truncate in tables:
        c = conn_check.cursor()
        c.execute(f'DELETE FROM {table_to_truncate}')
        conn_check.commit()

    conn_check.close()

<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Case Overview

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 💎 Learning Objectives

- Design and create a database for relational data.
    - Write basic SQL queries to create tables.
    - Learn how data should be stored to reduce or eliminate redundancy.
- Extract information from a database for financial reporting.
    - Retrieve data from one table.
    - Merge data from multiple tables.
    - Aggregate and filter data from one or more tables.
- Compute financial statement line items from transaction-level data.
- Integrate database skills with Python skills by using the `sqlite3` and `pandas` packages.

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

###  📑  Introduction

- In this case, you will assume the role of a preparer of financial statements for a fictitious company named **Northwind Traders**.
- You will create database tables to store customer information and customers' orders.
- You will also create a table to store information about the company's products.
- You will create a database to store transaction-level data.
- Finally, you will retrieve data from these tables, merge that data, and transform it into useful numbers like revenue or cost of goods sold.

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📏 Introduction to Databases

- A database is a collection of data. There are many different types of databases.
- SQL vs NoSQL:
    - **SQL** Database: A relational database consisted of tables
        - <span style="color: orange;">Common Brands</span>: Oracle DBMS, PostgreSQL, MySQL, SQL Server, SQLite
    - **NoSQL** Database: A non-relational database consisted of *documents*, key-value pairs, or graphs.
        - <span style="color: orange;">Common Brands</span>: MongoDB, Redis, Amazon DynamoDB
- Client-server vs Embedded:
    - **Client-server Model**: The database is located in a server (e.g., Amazon Web Services).
    - **Embedded**: Integrated within the apps (e.g., SQLite in Android apps).
- In this case study, we will use **SQLite**, an embedded SQL database.
- How is a database table different from a spreadsheet?
    - A column (called a field) in a database table *must* contain a homogeneous type of data.
    - In a database, a *primary key* 🔑 is used to identify each row (called a record).
    - You can define relationships between tables using one or more *foreign keys* in a database.

<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 1: Setting Up

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 1A: Place the CSV files into the same folder

#### 👇 Tasks

- ✔️ Place the 4 CSV files - `Customers.csv`, `Order Details.csv`, `Orders.csv`, `Products.csv` into the same folder as this Jupyter notebook. Do **not** modify the content of the original CSV files.
- ✔️ The file names should be:
  - `Customers.csv`
  - `Order Details.csv`
  - `Orders.csv`
  - `Products.csv`

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure your file names are correct.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, rename your files to match the ones listed above.</span>

In [4]:
# DO NOT CHANGE THE CODE IN THIS CELL
user_csv_filenames = glob.glob('*.csv')
required_filenames = [
    'Customers.csv',
    'Order Details.csv',
    'Orders.csv',
    'Products.csv',
]

# Check if all files exist in the current directory
tc.assertEqual(set(user_csv_filenames).intersection(required_filenames), set(required_filenames), 'Check the names and locations of the CSV files')

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 1B: Download and install DB Browser for SQLite

#### 👇 Tasks

- ✔️ Download [DB Browser for SQLite](https://sqlitebrowser.org/dl/), which is available for both Windows and Mac.
- ✔️ You can view the [installation guide here](https://www.notion.so/accy575/Installing-DB-Browser-for-SQLite-de58cee24caa4257924ba6eeadb1e487).

<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 2: Creating and Populating the Database

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 2A: Create `NWT.db` and create a table named `tblProducts`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ Once installation is complete, open DB Browser for SQLite.
- ✔️ Click on the <em style="color: BlueViolet;">New Database</em> button, marked by a green box in the image below.

![New Database](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_new_database.png)

- ✔️ **Navigate to the folder that contains this Jupyter notebook.**
- ✔️ Save your database file as `NWT.db` (this file must be saved in the same folder as this Jupyter notebook).
- ✔️ A new dialogue will appear that will allow you to create a new table. If it does not, click the <em style="color: BlueViolet;">Create Table</em> button.
- ✔️ Name your new table `tblProducts`, then click the <em style="color: BlueViolet;">Add</em> button.

<img src="https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_add_field.png" alt="Add field" width="500" />

- ✔️ Follow the steps below to create the `tblProducts` table.
    1. Replace the text <em style="color: BlueViolet;">Field1</em> with <em style="color: BlueViolet;">ProductCode</em>. Change the type to <em style="color: BlueViolet;">TEXT</em>, and select the <em style="color: BlueViolet;">PK</em> checkbox (this tells SQLite that <em style="color: BlueViolet;">ProductCode</em> will be your primary key column).
    2. Continue to click the <em style="color: BlueViolet;">Add</em> button and populate the dialog with all the fields shown in the image below.
    3. Make sure the <em style="color: BlueViolet;">PK</em> checkbox is checked for the <em style="color: BlueViolet;">ProductCode</em> field (and only for that field!).
    4. The field names must match the header row in the `Product.csv` file - `ProductCode`, `ProductName`, `StandardCost`, `ListPrice`, `QtyPerUnit`, `Category`.
    5. Note that, as you add fields, the bottom panel of the dialog updates to show the SQL code that is generated. Ultimately, DB Browser will execute SQL code to create a new table. The graphical user interface automates the process of writing SQL.
    6. When you have populated the dialog with all of the fields shown in the image below, click the <em style="color: BlueViolet;">OK</em> button in the bottom right.

<img src="https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblProducts_create.png" alt="Add field" width="600" />

- ✔️ In the main window for DB Browser, click the <em style="color: BlueViolet;">Write Changes</em> button. This will save the new table to the database file. **Don't skip this step!!**

<img src="https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblProducts_write_changes.png" alt="Write Changes" width="550" />

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure you have:
  1. Created the `NWT.db` file in the same folder as this Jupyter notebook.
  2. Correctly created an empty `tblProducts` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [5]:
# DO NOT CHANGE THE CODE IN THIS CELL
user_db_files = glob.glob('*.db')

# Check if all files exist in the current directory
tc.assertTrue('NWT.db' in user_db_files, f'Check if NWT.db exists in {os.getcwd()}')
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblProducts'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns and primary key
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value'])
df_correct_columns = pd.DataFrame({ 'name': ['ProductCode', 'ProductName', 'StandardCost', 'ListPrice', 'QtyPerUnit', 'Category'],
                                   'type': ['TEXT', 'TEXT', 'REAL', 'REAL', 'TEXT', 'TEXT'],
                                   'pk': [1, 0, 0, 0, 0, 0]})

pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 2B: Add data to the `tblProducts` table

You have now created a database file and, within it, a table that can store information about the company’s products. However, the table is empty. It has a structure, but no records. Thus, your next task is to populate the table with data from the file `Products.csv` that accompanies the case. Here, we'll use the `pandas` and `sqlite3` Python packages.

#### 👇 Tasks 

- ✔️ Read `Products.csv` to a new DataFrame named `df_products_raw`.
    - Only supply the file name to `read_csv()` without additional parameters (e.g., `pd.read_csv('Products.csv')`).
- ✔️ Use Pandas' `to_sql()` to populate the `tblProducts` table with the following parameters.
    - `name`: Name of the database table to add the data
    - `index`: Whether to add the index column to the database table
        - We don't want to add the index column - the index column in `df_products_csv` will simply be a range of integers)
    - `con`: The `Connection` object that represents the database. Use `conn`.
    - `if_exists`: How to behave if the table already exists.
        - Use `'append'`.
        - **Do NOT use `if_exists='replace'`.** Using `if_exists='replace'` will result in undesirable side effects.
- **Example**: `df.to_sql(name='myTableName', index=False, con=conn, if_exists='append')` appends data inside the `df` DataFrame to the `myTableName` database table.

#### 🚀 Helping you out...

- Running `to_sql(..., if_exists='append')` multiple times will continue to append rows to your database table. This means that you will end up with duplicates if you run your code more than once.
- Since you're working in a Jupyter notebook, you will likely run your `to_sql()` code multiple times.
- To prevent your database tables from accumulating duplicates, we provide the code below to delete all rows before appending new rows.

In [6]:
# DO NOT CHANGE THE CODE IN THIS CELL
# 🚀 Helping you out...
# Run this cell before you run your own code
# This prevents your database tables accumulating duplicate entries by emptying the table
def delete_all_rows_from_table(table_to_truncate):
    conn_check = sqlite3.connect('NWT.db')

    tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_check)['tbl_name'])
    if table_to_truncate in tables:
        c = conn_check.cursor()
        c.execute(f'DELETE FROM {table_to_truncate}')
        conn_check.commit()

    conn_check.close()
delete_all_rows_from_table('tblProducts')

In [7]:
conn = sqlite3.connect('NWT.db')

df_products_raw = pd.read_csv('Products.csv')
df_products_raw.to_sql('tblProducts', conn,if_exists='append',  index=False )



conn.close()

#### Check Your Work 🧭

- Once you're done, run the code cell below to ensure you have correctly populated the `tblProducts` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [8]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_products_check = pd.read_csv('https://raw.githubusercontent.com/accy575-uiuc/datasets/main/database-case/Products.csv')
df_products_db = pd.read_sql_query('SELECT * FROM tblProducts', con=conn_check)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_products_check.columns.sort_values().to_list()
df_products_check = df_products_check[new_column_order]
df_products_db = df_products_db[new_column_order]

pd.testing.assert_frame_equal(df_products_check.sort_values('ProductCode').reset_index(drop=True),
                             df_products_db.sort_values('ProductCode').reset_index(drop=True))

conn_check.close()

URLError: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 2C: Create `tblCustomers` table

Using the file `Customers.csv` that accompanies this case, create and populate a new table in the database, `tblCustomers`. Use your judgment in choosing data types for the columns and in choosing the primary key column(s).

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks 

- ✔️ Create a new table named `tblCustomers`.
- ✔️ Column names should match the header in `Customers.csv` (case-sensitive).
- ✔️ Each customer is identified by a unique `CustomerID`. Set `CustomerID` as a primary key.
- ✔️ `CustomerID` field should be an `INTEGER` type.
- ✔️ All other fields should be `TEXT` types.
    - Why does the `ZIP` field use `TEXT` type? Aren't ZIP codes all numbers? 🙄 We use `TEXT` type for two reasons:
        1. A zip code can be followed by 4 additional codes (e.g., 61821-0900).
        2. Some ZIP codes begin with 0 (e.g., 01003). If `INTEGER` type is used, it may be mistakenly be parsed as 1003).
        

- ✔️ Once you create the table, don't forget to click the <em style="color: BlueViolet;">Write Changes</em> button in the main window.

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [ ]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblCustomers'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns and primary key
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value'])
df_correct_columns = pd.DataFrame({ 'name': ['CustomerID', 'CustomerName', 'ContactName', 'ContactJobTitle', 'PhoneNum', 'Address', 'City', 'State', 'ZIP', 'Country'], 
                                   'type': ['INTEGER', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT'],
                                   'pk': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0] })
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 2D: Add data to the `tblCustomers` table

#### 👇 Tasks 

- ✔️ Read `Customers.csv` to a new DataFrame named `df_customers_raw`.
    - Only supply the file name to `read_csv()` without additional parameters (e.g., `pd.read_csv('Customers.csv')`).
- ✔️ Use Pandas' `to_sql()` to populate `tblCustomers` with data inside `df_customers_raw`.
    - **Do NOT use `if_exists='replace'`.** This will result in undesirable side effects.
    - **Example**: `df.to_sql(name='myTableName', index=False, con=conn, if_exists='append')` appends data inside the `df` DataFrame to the `myTableName` database table.

#### 🚀 Helping you out...

- To prevent your database tables from accumulating duplicates, we provide the code below to delete all rows before appending new rows.

In [9]:
# DO NOT CHANGE THE CODE IN THIS CELL
# 🚀 Helping you out...
# Run this cell before you run your own code
# This prevents your database tables accumulating duplicate entries by emptying the table
def delete_all_rows_from_table(table_to_truncate):
    conn_check = sqlite3.connect('NWT.db')

    tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_check)['tbl_name'])
    if table_to_truncate in tables:
        c = conn_check.cursor()
        c.execute(f'DELETE FROM {table_to_truncate}')
        conn_check.commit()

    conn_check.close()
delete_all_rows_from_table('tblCustomers')

In [10]:
conn = sqlite3.connect('NWT.db')

# YOUR CODE BEGINS
df_customers_raw = pd.read_csv('Customers.csv')
df_customers_raw.to_sql('tblCustomers', conn,if_exists='append',  index=False )

# YOUR CODE ENDS

conn.close()

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure you have correctly populated the `tblCustomers` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [11]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_customers_check = pd.read_csv('https://raw.githubusercontent.com/accy575-uiuc/datasets/main/database-case/Customers.csv',
                                dtype={ 'ZIP': str })
df_customers_db = pd.read_sql_query('SELECT * FROM tblCustomers', con=conn_check)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_customers_check.columns.sort_values().to_list()
df_customers_check = df_customers_check[new_column_order]
df_customers_db = df_customers_db[new_column_order]

pd.testing.assert_frame_equal(df_customers_check.sort_values('CustomerID').reset_index(drop=True),
                             df_customers_db.sort_values('CustomerID').reset_index(drop=True))

conn_check.close()

URLError: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>

<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 3: Tracking Customer Orders

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3A: Create a New Table named `tblOrders1`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ Create a new empty table in your database named `tblOrders1`.
- ✔️ Within that table, create the fields as shown in the screenshot below.
    - Remember: do not use spaces in column names.
    - Use the same data types as shown in the screenshot.
- ✔️ Do NOT set a primary key in this part.

![tblOrders1 DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders1.png)

- ✔️ Once you create the table, don't forget to click the <em style="color: BlueViolet;">Write Changes</em> button in the main window.

![tblOrders1 DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders1_write_changes.png)

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [12]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblOrders1'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value']) \

# Exclude Order ID
df_user_columns = df_user_columns[df_user_columns['name'] != 'OrderID']

df_correct_columns = pd.DataFrame({'name': ['CustomerName', 'OrderDate', 'Price', 'Product', 'Quantity', 'ShipDate'],
                                   'type': ['TEXT', 'TEXT', 'REAL', 'TEXT', 'INTEGER', 'TEXT'],
                                   'pk': [0, 0, 0, 0, 0, 0]})
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3B: Add an `OrderID` field to `tblOrders1`

Are we missing something? 🤨 Yes! With the current table structure of `tblOrders1`, we cannot uniquely identify an order. One possible solution is to set the primary key to two existing columns, the order date and the customer name. However, that would only work if customers never place two orders on the same day.

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ Create a separate field called `OrderID` with data type `INTEGER`.
- ✔️ When you click on the <em style="color: BlueViolet;">Add</em> button to create a new field, it will add a new row to the bottom. Use <em style="color: BlueViolet;">Move to top</em> button to move the new `OrderID` field to the top.
- ✔️ Mark the column as a primary key (<span style="color: blue;">PK checkbox in the screenshot</span>).
- ✔️ Also add the auto-increment attribute (<span style="color: red;">AI checkbox in the screenshot</span>).
    - The auto increment attribute tells the database to automatically generate a new order ID every time a new row is added. Neat, huh?

![tblOrders1 new primary key](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders1_order_id_field.png)

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [13]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblOrders1'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value'])
df_correct_columns = pd.DataFrame({'name': ['CustomerName', 'OrderDate', 'OrderID', 'Price', 'Product', 'Quantity', 'ShipDate'],
                                   'type': ['TEXT', 'TEXT', 'INTEGER', 'REAL', 'TEXT', 'INTEGER', 'TEXT'], 
                                   'pk': [0, 0, 1, 0, 0, 0, 0]}) \
    
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3C: Populate `tblOrders1` with sample data

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ Populate `tblOrders1` with the three orders shown in the table below using the DB Browser.
    - Although the `OrderID` values shown in the table are `1`, `2`, `3`, it is acceptable to use different `OrderID` values (e.g., `10`, `14`, `17`) as long as they are unique.

|     OrderID    |     Order Date    |     Customer Name    |     Product                          |     Quantity    |     Price    |     Ship Date     |
|----------------|-------------------|----------------------|--------------------------------------|-----------------|--------------|-------------------|
|     1          |     2018-10-10    |     Company AA       |     Northwind Traders   Chai         |     100         |     18.00    |     2018-10-10    |
|     2          |     2018-10-11    |     Company   AA     |     Northwind   Traders Beer         |     500         |     14.00    |     2018-10-11    |
|     3          |     2018-10-11    |     Company H        |     Northwind Traders   Olive Oil    |     200         |     21.35    |     2018-10-11    |

- ✔️ To insert new rows, go to <em style="color: BlueViolet;">Browse Data</em> tab and select `tblOrders1` table.
- ✔️ Then, click on the icon with a <span style="color: green;">+</span> mark to add new row.
- ✔️ Fill in the cells.

![Populate tblOrders1](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_populate_tblOrders1.png)

- ✔️ Once you finish filling in the three new rows, click on <em style="color: BlueViolet;">Write Changes</em>.

![Populate tblOrders1 Write Changes](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_populate_tblOrders1_write_changes.png)

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure you have correctly populated the `tblOrders1` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [14]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_orders1_check = pd.DataFrame({ 'OrderDate': ['2018-10-10', '2018-10-11', '2018-10-11'],
 'CustomerName': ['Company AA', 'Company AA', 'Company H'],
 'Product': ['Northwind Traders Chai', 'Northwind Traders Beer', 'Northwind Traders Olive Oil'],
 'Quantity': [100, 500, 200], 'Price': [18.0, 14.0, 21.35],
 'ShipDate': ['2018-10-10', '2018-10-11', '2018-10-11']});
df_orders1_db = pd.read_sql_query('SELECT * FROM tblOrders1', con=conn_check)

print('Your tblOrders1 table:')
display(df_orders1_db)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_orders1_check.columns.sort_values().to_list()
df_orders1_check = df_orders1_check[new_column_order]
df_orders1_db = df_orders1_db[new_column_order]

pd.testing.assert_frame_equal(df_orders1_db.sort_values('Quantity').reset_index(drop=True),
                             df_orders1_check.sort_values('Quantity').reset_index(drop=True))

conn_check.close()

Your tblOrders1 table:


,OrderID,OrderDate,CustomerName,Product,Quantity,Price,ShipDate
0,1,2018-10-10,Company AA,Northwind Traders Chai,100,18.00,2018-10-10
1,2,2018-10-11,Company AA,Northwind Traders Beer,500,14.00,2018-10-11
2,3,2018-10-11,Company H,Northwind Traders Olive Oil,200,21.35,2018-10-11


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3D: Product and customer names as strings?

#### 👇 Tasks

- ✔️ <span style="color: blue;">In a few sentences</span>, explain why storing the product and customer names as text strings in the orders table is inefficient, unwise, and downright stupid.
- ✔️ What can go wrong with this design?
- ✔️ Use the markdown cell below.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here...Lack of standards: It does not offer a standard way to specify data format. No standard way to express “special characters”.........

Inefficiency: It is most likely to lead to massive redundancy (repetition of values). Speed of access and space efficiency for large data sets. Difficult to store “non-rectangular” data sets. Internationalization.........

Lack of data integrity: lack of data integrity measures- Figures are prone to manipulation
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3E: Can you think of a better design?

#### 👇 Tasks

- ✔️ <span style="color: blue;">In a few sentences</span>, propose a better design for incorporating the product and customer information into the orders table.
- ✔️ Use the markdown cell below.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here... both the customer and product information should be set as unique values so as to prevent the redundancies or repetitions of the product and customer information
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3F: Create `tblOrders2`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### ⚙️ Check Settings

- Let's try a different design for our orders table.
- However, we need you to check a setting in DB Browser.
- Click on the tab titled <em style="color: BlueViolet;">Edit Pragmas</em>, and make sure that <em style="color: BlueViolet;">Foreign Keys</em> is checked.

![tblOrders1 DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_pragma_foreign_keys.png)

#### 👇 Tasks

- ✔️ You will now create a new database table, `tblOrders2`.
- ✔️ To save you the grunt work of entering in the details of each column, we will (finally!) use SQL code.
- ✔️ Go to <em style="color: BlueViolet;">Execute SQL</em> tab.
- ✔️ Copy & paste the SQL code below (`CREATE TABLE ...`).
- ✔️ Run the SQL code by clicking on the <em style="color: BlueViolet;">Execute all/selected SQL</em> icon button.
- ✔️ Refer to the screenshot for an example.

#### 📄 SQL Code

```sql
CREATE TABLE `tblOrders2` (
  `OrderID` INTEGER PRIMARY KEY AUTOINCREMENT, 
  `OrderDate` TEXT, 
  `CustomerID` INTEGER, 
  `ProductCode` TEXT, 
  `Quantity` INTEGER, 
  `Price` REAL, 
  `ShipDate` TEXT, 
  FOREIGN KEY(`CustomerID`) REFERENCES `tblCustomers`(`CustomerID`), 
  FOREIGN KEY(`ProductCode`) REFERENCES `tblProducts`(`ProductCode`)
);
```

![Execute SQL](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders2_execute_sql.png)

- ✔️ Check if you see an <code style="color: green;">Execution finished without errors.</code> message at the bottom.

![Execute SQL Result](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders2_execute_sql_result.png)

- ✔️ Finally, be sure to <em style="color: BlueViolet;">Write Changes</em> once you run the SQL code.

![Write Changes](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_write_result_button.png)

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [15]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblOrders2'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns and primary key
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value']) \

df_correct_columns = pd.DataFrame({'name': ['CustomerID', 'OrderDate', 'OrderID', 'Price', 'ProductCode', 'Quantity', 'ShipDate'], 
                                   'type': ['INTEGER', 'TEXT', 'INTEGER', 'REAL', 'TEXT', 'INTEGER', 'TEXT'], 
                                   'pk': [0, 0, 1, 0, 0, 0, 0]})
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

# Check foreign keys
df_user_foreign_keys = pd.read_sql_query(f'PRAGMA foreign_key_list("{table_to_check}");', conn_checker)[['table', 'from', 'to']]
df_correct_foreign_keys = pd.DataFrame({ 'table': ['tblProducts', 'tblCustomers'],
                                        'from': ['ProductCode', 'CustomerID'],
                                        'to': ['ProductCode', 'CustomerID'] })

pd.testing.assert_frame_equal(df_user_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True),
                              df_correct_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3G: Try to insert a new row

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ The SQL code below (`INSERT INTO ...`) attempts to insert a row into `tblOrders2`.
- ✔️ Create a new tab in <em style="color: BlueViolet;">Execute SQL</em> window using the leftmost icon button.
- ✔️ Type this SQL code into the new tab and run it.

##### 📄 SQL Code
```sql
INSERT INTO tblOrders2 (OrderDate, CustomerID, ProductCode, Quantity, Price, ShipDate)
VALUES ('2018-10-10', 'Company AA', 'Northwind Traders Chai', 100, 18.00, '2018-10-10');
```

![Execute SQL](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_insert_into_tblorders2_attempt.png)

- ✔️ **You will receive an error.**
- ✔️ <span style="color: blue;">In a paragraph</span>, explain the following:
    1. What is the error?
        - You can find this in the second line of DB Browser's output message (e.g., <code style="background: #ff5544; color: white;">Result: UNIQUE constraint failed</code>).
    2. What does it mean?
    3. Why did you receive it?
- ✔️ <span style="color: red;">Cite your source(s) if you are borrowing an idea from an external link.</span>
    - It can be an informal citation (e.g., A URL).
- ✔️ Use the markdown cell below.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here...The error states that... FOREIGN KEY Constraint failed ...
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3H: Populate `tblOrders2`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks

- ✔️ Populate `tblOrders2` with the three orders shown in the table below using the DB Browser.
- ✔️ When inserting these rows in DB Browser, substitute the customer name in `Customer Name` column with an appropriate `CustomerID` value from `tblCustomers`.
    - **Example**: Assume you are given a row with a customer name `Company B`. To find the `CustomerID` of `Company B`, locate the row for `Company B` in `tblCustomers`. `Company B` has a `CustomerID` of `2`. You should use `2` instead of `Customer B` for `tblOrder2`'s `CustomerID` column.
- ✔️ Similarly, substitute the product name in `Product` column with an appropriate `ProductCode` value from `tblProducts`.
- ✔️ You are free to use either the <em style="color: BlueViolet;">Browse Data</em> or <em style="color: BlueViolet;">Execute SQL</em> tab.
  - We will only check your result, not how you did it.

|     OrderID    |     Order Date    |     Customer Name    |     Product                          |     Quantity    |     Price    |     Ship Date     |
|----------------|-------------------|----------------------|--------------------------------------|-----------------|--------------|-------------------|
|     1          |     2018-10-10    |     Company AA       |     Northwind Traders   Chai         |     100         |     18.00    |     2018-10-10    |
|     2          |     2018-10-11    |     Company   AA     |     Northwind   Traders Beer         |     500         |     14.00    |     2018-10-11    |
|     3          |     2018-10-11    |     Company H        |     Northwind Traders   Olive Oil    |     200         |     21.35    |     2018-10-11    |

- ✔️ Once you create the table, don't forget to click the <em style="color: BlueViolet;">Write Changes</em> button in the main window.

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [16]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_orders2_check = pd.DataFrame({ 'OrderID': [1, 2, 3], 'OrderDate': ['2018-10-10', '2018-10-11', '2018-10-11'], 
                                 'CustomerID': [27, 27, 8], 'ProductCode': ['NWTB-1', 'NWTB-34', 'NWTO-5'], 
                                 'Quantity': [100, 500, 200], 'Price': [18.0, 14.0, 21.35], 
                                 'ShipDate': ['2018-10-10', '2018-10-11', '2018-10-11'] });
df_orders2_db = pd.read_sql_query('SELECT * FROM tblOrders2', con=conn_check)

print('Your tblOrders2 table:')
display(df_orders2_db)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_orders2_check.columns.sort_values().to_list()
df_orders2_check = df_orders2_check[new_column_order]
df_orders2_db = df_orders2_db[new_column_order]

pd.testing.assert_frame_equal(df_orders2_db.drop(columns=['OrderID']).sort_values('Quantity').reset_index(drop=True),
                             df_orders2_check.drop(columns=['OrderID']).sort_values('Quantity').reset_index(drop=True))

conn_check.close()

Your tblOrders2 table:


,OrderID,OrderDate,CustomerID,ProductCode,Quantity,Price,ShipDate
0,1,2018-10-10,27,NWTB-1,100,18.00,2018-10-10
1,2,2018-10-11,27,NWTB-34,500,14.00,2018-10-11
2,3,2018-10-11,8,NWTO-5,200,21.35,2018-10-11


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3I: What is a foreign key?

#### 👇 Tasks

- ✔️ <span style="color: blue;">In a few sentences</span>, explain what a **foreign key** 🔑 is.
- ✔️ You will need to Google this.
- ✔️ <span style="color: red;">Cite your source(s) if you are borrowing an idea from an external link.</span>
    - It can be an informal citation (e.g., A URL).
- ✔️ Use the markdown cell below.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here...A FOREIGN Key is a field (or collection of fields) in one table, that refers to the PRIMARY KEY in another table
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3J: What is a foreign key constraint?

#### 👇 Tasks

- ✔️ <span style="color: blue;">In a few sentences</span>, explain what a **foreign key constraint** is.
- ✔️ <span style="color: red;">Cite your source(s) if you are borrowing an idea from an external link.</span>
    - It can be an informal citation (e.g., A URL).
- ✔️ Use the markdown cell below.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here...A foreign key constraint specifies that the key can only contain values that are in the referenced primary key, and thus ensures the referential integrity of data that is joined on the two keys
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3K: Comparing `tblOrders2` with `tblOrders1`

#### 👇 Tasks

- ✔️ <span style="color: blue;">In a few sentences</span>, explain **why the design of `tblOrders2` is superior to that of `tblOrders1`**.
- ✔️ <span style="color: red;">Cite your source(s) if you are borrowing an idea from an external link.</span>
    - It can be an informal citation (e.g., A URL).
- ✔️ Use the markdown cell below.

#### 💡 Hint

- Read the section titled *What is good database design?* on [this web page](https://support.microsoft.com/en-us/office/database-design-basics-eb2159cf-1e30-401a-8084-bd4f9c9ca1f5?ui=en-us&rs=en-us&ad=us#bmgood).

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Enter your response here...

- Relationships with data from other tables is well defined in tblOrders2 than on tblOrders1

- In tblOrders2 data is more well normalized than in tblOrders1
</p>

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📐 Identify the problem with `tblOrders2`

Our second design of the orders table was, unfortunately, not sufficient. 😐 **It cannot handle a situation in which a customer orders multiple products in one order.** This is common in retailing. For example, think about your past orders from Amazon. Did you ever order more than one product?

How can we redesign our orders table to accommodate multiple products? The naïve solution is to add multiple product columns. But this is a poor design because it imposes a maximum order size. Even if you have 20 product columns, a customer might order 25 items.

A best practice is to create two tables to store orders. The first table, `tblOrders3`, will have one record per order. The second table, `tblOrderDetails`, will have one record per product per order. Thus, there will be a one-to-many relationship between `tblOrders3` and `tblOrderDetails`.

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3L: Create `tblOrders3`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks 

- ✔️ Create a new empty table named `tblOrders3`.
- ✔️ Use the column names used in `Orders.csv`.
- ✔️ Use `INTEGER` types for ID columns, and `TEXT` types for date columns.
- ✔️ Set an appropriate primary key.
    - Auto-increment attribute is optional - you can either check or uncheck it.
- ✔️ Add a foreign key constraint on `tblOrders3`'s `CustomerID` column to `tblCustomer`'s `CustomerID` column.
    - Refer to the screenshot below to see how you can add a foreign key when creating/modifying a table.
    - If you don't see a dropdown box for foreign keys, scroll to the right in <em style="color: BlueViolet;">Edit Table Definition</em> window and double-click on the blank area under *Foreign Key* header.
    - You'll have to make selections in two dropdown boxes.
        - <span style="color: blue;">First dropdown box</span>: name of the table to find the referenced column
        - <span style="color: blue;">Second dropdown box</span>: name of the referenced column
    - After making your selections, you'll need to click outside of the dropdowns before clicking "Okay" in order for your choices to take effect.

![Add a foreign key](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_tblOrders3_set_foreign_key.png)

- ✔️ Once you create the table, don't forget to click the <em style="color: BlueViolet;">Write Changes</em> button in the main window.

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [17]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblOrders3'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns and primary key
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value'])
df_correct_columns = pd.DataFrame({ 'name': ['OrderID', 'CustomerID', 'OrderDate', 'ShipDate'],
                                   'type': ['INTEGER', 'INTEGER', 'TEXT', 'TEXT'], 
                                   'pk': [1, 0, 0, 0]})
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

# Check foreign keys
df_user_foreign_keys = pd.read_sql_query(f'PRAGMA foreign_key_list("{table_to_check}");', conn_checker)[['table', 'from', 'to']]
df_correct_foreign_keys = pd.DataFrame({ 'table': ['tblCustomers'], 
                                        'from': ['CustomerID'], 
                                        'to': ['CustomerID']})

pd.testing.assert_frame_equal(df_user_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True),
                              df_correct_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3M: Populate `tblOrders3` with `Orders.csv`

#### 👇 Tasks 

- ✔️ Read `Orders.csv` to a new DataFrame named `df_orders_raw`.
    - Only supply the file name to `read_csv()` without additional parameters (e.g., `pd.read_csv('Orders.csv')`).
- ✔️ Use Pandas' `to_sql()` to populate `tblOrders3` with data inside `df_orders_raw`.
    - **Do NOT use `if_exists='replace'`.** Using `if_exists='replace'` will result in undesirable side effects.
    - **Example**: `df.to_sql(name='myTableName', index=False, con=conn, if_exists='append')` appends data inside the `df` DataFrame to the `myTableName` database table.

#### 🚀 Helping you out...

- To prevent your database tables from accumulating duplicates, we provide the code below to delete all rows before appending new rows.

In [18]:
# DO NOT CHANGE THE CODE IN THIS CELL
# 🚀 Helping you out...
# Run this cell before you run your own code
# This prevents your database tables accumulating duplicate entries by emptying the table
def delete_all_rows_from_table(table_to_truncate):
    conn_check = sqlite3.connect('NWT.db')

    tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_check)['tbl_name'])
    if table_to_truncate in tables:
        c = conn_check.cursor()
        c.execute(f'DELETE FROM {table_to_truncate}')
        conn_check.commit()

    conn_check.close()
delete_all_rows_from_table('tblOrders3')

In [19]:
conn = sqlite3.connect('NWT.db')

# YOUR CODE BEGINS
df_orders_raw = pd.read_csv('Orders.csv')
df_orders_raw.to_sql('tblOrders3', conn, if_exists='append',  index=False )

# YOUR CODE ENDS

conn.close()

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure you have correctly populated the `tblProducts` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [20]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_orders_check = pd.read_csv('https://raw.githubusercontent.com/accy575-uiuc/datasets/main/database-case/Orders.csv')
df_orders3_db = pd.read_sql_query('SELECT * FROM tblOrders3', con=conn_check)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_orders_check.columns.sort_values().to_list()
df_orders_check = df_orders_check[new_column_order]
df_orders3_db = df_orders3_db[new_column_order]

pd.testing.assert_frame_equal(df_orders_check.sort_values('OrderID').reset_index(drop=True),
                             df_orders3_db.sort_values('OrderID').reset_index(drop=True))

conn_check.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3N: Create `tblOrderDetails`

![Use DB Browser](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/dbbrowser_use_dbbrowser.png)

#### 👇 Tasks 

- ✔️ Create a new empty table named `tblOrderDetails`.
- ✔️ Use the column names used in `Order Details.csv`.
- ✔️ Use the following data types:
    - `ID`: `INTEGER`
    - `OrderID`: `INTEGER`
    - `ProductCode`: `TEXT`
    - `Quantity`: `INTEGER`
    - `Price`: `REAL`
- ✔️ Set an appropriate primary key.
    - Auto-increment attribute is optional - you can either check or uncheck it.
- ✔️ Add the following foreign key constraints:
    - `tblOrderDetails`'s `OrderID` column to `tblOrders3`' `OrderID` column.
    - `tblOrderDetails`'s `ProductCode` column to `tblProducts`' `ProductCode` column.
- ✔️ Refer to the previous screenshots if you're stuck.

#### 🧭 Check Your Work

- Once you're done, run the code cell below to check your work.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [21]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_checker = sqlite3.connect('NWT.db')
table_to_check = 'tblOrderDetails'

# Check if table exists
user_tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_checker)['tbl_name'])
tc.assertTrue(table_to_check in user_tables, f'{table_to_check} does not exist in your NWT.db file!')

# Check columns and primary key
# Ignore column ID, notnull, default value settings
df_user_columns = pd.read_sql_query(f'PRAGMA table_info("{table_to_check}");', conn_checker) \
    .drop(columns=['cid', 'notnull', 'dflt_value'])
df_correct_columns = pd.DataFrame({ 'name': ['ID', 'OrderID', 'ProductCode', 'Quantity', 'Price'], 
                                   'type': ['INTEGER', 'INTEGER', 'TEXT', 'INTEGER', 'REAL'], 
                                   'pk': [1, 0, 0, 0, 0] })
pd.testing.assert_frame_equal(df_user_columns.sort_values('name').reset_index(drop=True),
                              df_correct_columns.sort_values('name').reset_index(drop=True))

# Check foreign keys
df_user_foreign_keys = pd.read_sql_query(f'PRAGMA foreign_key_list("{table_to_check}");', conn_checker)[['table', 'from', 'to']]
df_correct_foreign_keys = pd.DataFrame({ 'table': ['tblProducts', 'tblOrders3'],
                                         'from': ['ProductCode', 'OrderID'],
                                         'to': ['ProductCode', 'OrderID'] })
pd.testing.assert_frame_equal(df_user_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True),
                              df_correct_foreign_keys.sort_values(['table', 'from', 'to']).reset_index(drop=True))

conn_checker.close()

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 3O: Populate `tblOrderDetails` with `Order Details.csv`

#### 👇 Tasks 

- ✔️ Read `Order Details.csv` to a new DataFrame named `df_order_details_raw`.
    - Only supply the file name to `read_csv()` without additional parameters (e.g., `pd.read_csv('Order Details.csv')`).
- ✔️ Use Pandas' `to_sql()` to populate `tblOrderDetails` with data inside `df_order_details_raw`.
    - **Do NOT use `if_exists='replace'`.** Using `if_exists='replace'` will result in undesirable side effects.
    - **Example**: `df.to_sql(name='myTableName', index=False, con=conn, if_exists='append')` appends data inside the `df` DataFrame to the `myTableName` database table.

#### 🚀 Helping you out...

- To prevent your database tables from accumulating duplicates, we provide the code below to delete all rows before appending new rows.

In [22]:
# DO NOT CHANGE THE CODE IN THIS CELL
# 🚀 Helping you out...
# Run this cell before you run your own code
# This prevents your database tables accumulating duplicate entries by emptying the table
def delete_all_rows_from_table(table_to_truncate):
    conn_check = sqlite3.connect('NWT.db')

    tables = list(pd.read_sql_query('SELECT * FROM sqlite_master WHERE type="table";', con=conn_check)['tbl_name'])
    if table_to_truncate in tables:
        c = conn_check.cursor()
        c.execute(f'DELETE FROM {table_to_truncate}')
        conn_check.commit()

    conn_check.close()
delete_all_rows_from_table('tblOrderDetails')

In [23]:
conn = sqlite3.connect('NWT.db')

# YOUR CODE BEGINS
df_order_details_raw = pd.read_csv('Order Details.csv')
df_order_details_raw.to_sql('tblOrderDetails', conn,if_exists='append',  index=False )
# YOUR CODE ENDS

conn.close()

#### 🧭 Check Your Work

- Once you're done, run the code cell below to ensure you have correctly populated the `tblProducts` table.
- <span style="color: green;">If the code cell runs without an error, you're good to move on.</span>
- <span style="color: red;">If the code cell throws an error, go back and check if you've missed any step.</span>

In [24]:
# DO NOT CHANGE THE CODE IN THIS CELL
conn_check = sqlite3.connect('NWT.db')

df_order_details_check = pd.read_csv('https://raw.githubusercontent.com/accy575-uiuc/datasets/main/database-case/Order%20Details.csv')
df_order_details_db = pd.read_sql_query('SELECT * FROM tblOrderDetails', con=conn_check)

# Reorder columns in alphabetical order to allow different column orders
new_column_order = df_order_details_check.columns.sort_values().to_list()
df_order_details_check = df_order_details_check[new_column_order]
df_order_details_db = df_order_details_db[new_column_order]

pd.testing.assert_frame_equal(df_order_details_check.sort_values('ID').reset_index(drop=True),
                             df_order_details_db.sort_values('ID').reset_index(drop=True))

conn_check.close()

<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 4: Extracting Information from the Database using Queries

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📐 A primer on SQL queries

The SQL `SELECT` statement is the most useful piece of SQL you can learn. The basic syntax is:

```sql
SELECT column1, column2 FROM table_name;
```

Executing the SQL query above retrieves *all rows* from `column1` and `column2` in `table_name`. It is also possible to calculate values in a `SELECT` statement. As an example, the query below will return the margin for every product in `tblProducts`.

```sql
SELECT ListPrice - StandardCost FROM tblProducts;
```

Run the code cell below to run see the query in action. 🔥

<span style="color: blue;"><strong>Note</strong>: The triple quotes (`'''` or `"""`) used in `stmt_margin1` are *multiline strings*.</span>

In [25]:
# DO NOT CHANGE THE CODE IN THIS CELL
# This is an example of a SQL query
stmt_margin1 = '''
SELECT ListPrice - StandardCost
FROM tblProducts;
'''

conn = sqlite3.connect('NWT.db') # Create a connection object to NWT.db
df_margin1 = pd.read_sql_query(stmt_margin1, con=conn) # Run statement and retrieve results as a DataFrame
display(df_margin1.head(5))
conn.close() # Close connection

,ListPrice - StandardCost
0,4.50
1,3.50
2,11.50
3,0.99
4,2.00


This output above isn't too useful. We cannot see the products associated with each row. A more useful query would be:

```sql
SELECT ProductCode, ProductName, (ListPrice - StandardCost) AS ContributionMargin
FROM tblProducts;
```

Run the code cell below to run see the query in action. 🔥

In [26]:
# DO NOT CHANGE THE CODE IN THIS CELL
# This is an example of a SQL query
stmt_margin2 = '''
SELECT ProductCode, ProductName, (ListPrice - StandardCost) AS ContributionMargin
FROM tblProducts;
'''

conn = sqlite3.connect('NWT.db') # Create a connection object to NWT.db
df_margin2 = pd.read_sql_query(stmt_margin2, con=conn) # Run statement and retrieve results as a DataFrame
display(df_margin2.head(5))
conn.close() # Close connection

,ProductCode,ProductName,ContributionMargin
0,NWTB-1,Northwind Traders Chai,4.50
1,NWTB-34,Northwind Traders Beer,3.50
2,NWTB-43,Northwind Traders Coffee,11.50
3,NWTB-81,Northwind Traders Green Tea,0.99
4,NWTB-87,Northwind Traders Tea,2.00


Much better ✨! Notice that I've used the `AS` keyword to rename the computed column. Finally, we can add a `WHERE` clause to selectively find rows matching one or more condition.

```sql
SELECT ProductCode, ProductName, (ListPrice - StandardCost) AS ContributionMargin
FROM tblProducts
WHERE ContributionMargin > 10.00;
```

Run the code cell below to run see the query in action. 🔥

In [27]:
# DO NOT CHANGE THE CODE IN THIS CELL
# This is an example of a SQL query
stmt_margin3 = '''
SELECT ProductCode, ProductName, (ListPrice - StandardCost) AS ContributionMargin
FROM tblProducts
WHERE ContributionMargin > 10.00;
'''

conn = sqlite3.connect('NWT.db') # Create a connection object to NWT.db
df_margin3 = pd.read_sql_query(stmt_margin3, con=conn) # Run statement and retrieve results as a DataFrame
display(df_margin3.head(5))
conn.close() # Close connection

,ProductCode,ProductName,ContributionMargin
0,NWTB-43,Northwind Traders Coffee,11.50
1,NWTDFN-51,Northwind Traders Dried Apples,13.25
2,NWTJP-7,Northwind Traders Marmalade,20.25


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 4A: Customers in the state of Washington

#### 👇 Tasks 

- ✔️ Write a SQL `SELECT` statement that retrieves **all columns** for all customers located in Washington state from `tblCustomers`.
- ✔️ Store your query in a new variable named `query_4a`.
- ✔️ `pd.read_sql_query()` executes a SQL query and retrieves the result as a Pandas Dataframe.
- ✔️ We'll use `df_4a` to grade your work.
    - The columns you retrieve can be in any order.
    - You do not need to sort `df_4a`.

#### 💡 Hint

- `SELECT * FROM ...` will retrieve **all columns**.

In [28]:
# YOUR CODE BEGINS
query_4a = '''
SELECT * FROM tblCustomers
WHERE City = 'Washington' ;
'''


# YOUR CODE ENDS

conn = sqlite3.connect('NWT.db')
df_4a = pd.read_sql_query(query_4a, con=conn)
display(df_4a)
conn.close()

,CustomerID,CustomerName,ContactName,ContactJobTitle,PhoneNum,Address,City,State,ZIP,Country


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 4B: Customers in either New York or Minnesota

#### 👇 Tasks 

- ✔️ Write a SQL `SELECT` statement that retrieves **all columns** for all customers located in **either** New York or Minnesota from `tblCustomers`.
- ✔️ Store your query in a new variable named `query_4b`.
- ✔️ We'll use `df_4b` to grade your work.
    - The columns you retrieve can be in any order.
    - You do not need to sort `df_4b`.

#### 💡 Hint

- Check out the SQL `IN` operator.

In [109]:
# YOUR CODE BEGINS
query_4b = '''
SELECT * FROM tblCustomers
WHERE City = 'New York' OR ' Minnesota';
'''


# YOUR CODE ENDS

conn = sqlite3.connect('NWT.db')
df_4b = pd.read_sql_query(query_4b, con=conn)
display(df_4b)
conn.close()

,CustomerID,CustomerName,ContactName,ContactJobTitle,PhoneNum,Address,City,State,ZIP,Country
0,4,Company D,Christina Lee,Purchasing Manager,(123)555-0100,123 4th Street,New York,NY,99999,USA
1,20,Company T,George Li,Purchasing Manager,(123)555-0100,789 20th Street,New York,NY,99999,USA


<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 5: Merging Data

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📐 A primer on SQL joins

Assume you are given the following tables.

![Two Tables Before Merge](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/merge_tutorial_tbl_profs_zip_desirability.png)

**Question**: Who lives in a *High*ly desirable ZIP code?

You can first locate a row in `tblDesirability` with a *High*  desirability. Then, you would have to look up a name with the corresponding ZIP code. This is an inefficient and inconvenient approach. As a student taking this course, you shouldn't even consider this approach. 👻

A much better way is to *join* the two tables. You can do this with a SQL `JOIN` clause, or with Pandas' `pd.merge()` method. The output is dependent on what *type* of join you use.

![Two Tables After Join](https://accy575-sp2021-course-assets.s3-us-west-2.amazonaws.com/images/merge_tutorial_tbl_profs_zip_desirability_joined.png)

**Wait, why is Mike missing in the inner join result?**

Poor Mike 😲! An inner join will only return rows that have matching counterparts in the other table. Mike's 85721 ZIP code is not listed on the `tblDesirability` table.

If you want to show everyone in the `tblProfs` table regardless of whether there is a matching ZIP code in `tblDesirability`, you can use a left join. A left join includes every row on the left, even if there is no match in the right. Since there's no matching ZIP code for Mike, his `Desirability` column will not contain a value (<code style="background: red; color: white;">NaN</code>).

We will first retrieve tables to Pandas DataFrames, and then use Pandas' `pd.merge()` method. `pd.merge()` essentially performs a SQL join.

Run the code below to first create `df_profs` and `df_desirability`. 🔥

In [30]:
# DO NOT CHANGE THE CODE IN THIS CELL
df_profs = pd.DataFrame({
    'Name': ['Vic', 'Kim', 'Josh', 'Wei', 'Mike'],
    'ZIP': ['02138', '98195', '61820', '06520', '85721']
})

df_desirability = pd.DataFrame({
    'ZIP': ['06520', '61820', '70210', '02138', '98195'],
    'Desirability': ['Low', 'Medium', 'Low', 'Medium', 'High']
})

print('df_profs:')
display(df_profs)

print('')
print('df_desirability:')
display(df_desirability)

df_profs:


,Name,ZIP
0,Vic,02138
1,Kim,98195
2,Josh,61820
3,Wei,06520
4,Mike,85721



df_desirability:


,ZIP,Desirability
0,06520,Low
1,61820,Medium
2,70210,Low
3,02138,Medium
4,98195,High


Run the code below to run a **inner join (merge)** with `df_profs` and `df_desirability`. 🔥

In [38]:
df_inner_merge = pd.merge(left=df_profs, right=df_desirability, how='inner', on='ZIP') 
display(df_inner_merge)

,Name,ZIP,Desirability
0,Vic,02138,Medium
1,Kim,98195,High
2,Josh,61820,Medium
3,Wei,06520,Low


Run the code below to run a **left join (merge)** with `df_profs` and `df_desirability`. 🔥

In [32]:
df_left_merge = pd.merge(left=df_profs, right=df_desirability, how='left', on='ZIP')
display(df_left_merge)

,Name,ZIP,Desirability
0,Vic,02138,Medium
1,Kim,98195,High
2,Josh,61820,Medium
3,Wei,06520,Low
4,Mike,85721,NaN


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5A: Retrieve database tables to DataFrames

#### 👇 Tasks 

- ✔️ Using `pd.read_sql_query()`, retrieve the following tables as DataFrames from `NWT.db`:
    - Table `tblCustomers` to a new variable named `df_customers`
    - Table `tblProducts` to a new variable named `df_products`
    - Table `tblOrders3` to a new variable named `df_orders`
        - 📅 **Convert `OrderDate` and `ShipDate` columns as datetime types**.
        - `display(df_orders[['OrderDate', 'ShipDate']].dtypes)` should return `datetime64[ns]` for both columns.
    - Table `tblOrderDetails` to a new variable named `df_order_details`

#### 💡 Hint

1. The code example below retrieves all rows & columns in the `tblCourses` table to `df_courses`.
    - **Example**: `df_courses = pd.read_sql_query('SELECT * FROM tblCourses', con=conn)`
    - In this example, the `conn` variable is a connection object linked to the database with the `tblCourses` table.
2. `pd.read_sql_query()` has an optional `parse_dates` parameter to parse specified columns as datetime types.
    - **Example**: `pd.read_sql_query('SELECT * FROM myTable', con=conn, parse_dates=["DateCol"])` parses `DateCol` column as a datetime type when reading query result.

In [31]:
conn = sqlite3.connect('NWT.db')

# YOUR CODE BEGINS
df_customers = pd.read_sql_query('SELECT * FROM tblCustomers', con=conn)
df_products = pd.read_sql_query('SELECT * FROM tblProducts', con=conn)
df_orders = pd.read_sql_query('SELECT * FROM tblOrders3', con=conn)
df_orders = df_orders[[ 'OrderDate', 'ShipDate']].apply(pd.to_datetime)
df_order_details = pd.read_sql_query('SELECT * FROM tblOrderDetails', con=conn)
# YOUR CODE ENDS

conn.close()
display(df_customers.head(5))
display(df_products.head(5))
display( df_orders[[ 'OrderDate', 'ShipDate']].apply(pd.to_datetime).head(5))
display(df_order_details.head(5))


,CustomerID,CustomerName,ContactName,ContactJobTitle,PhoneNum,Address,City,State,ZIP,Country
0,1,Company A,Anna Bedecs,Owner,(123)555-0100,123 1st Street,Seattle,WA,99999,USA
1,2,Company B,Antonio Gratacos Solsona,Owner,(123)555-0100,123 2nd Street,Boston,MA,99999,USA
2,3,Company C,Thomas Axen,Purchasing Representative,(123)555-0100,123 3rd Street,Los Angelas,CA,99999,USA
3,4,Company D,Christina Lee,Purchasing Manager,(123)555-0100,123 4th Street,New York,NY,99999,USA
4,5,Company E,Martin O’Donnell,Owner,(123)555-0100,123 5th Street,Minneapolis,MN,99999,USA


,ProductCode,ProductName,StandardCost,ListPrice,QtyPerUnit,Category
0,NWTB-1,Northwind Traders Chai,13.5,18.00,10 boxes x 20 bags,Beverages
1,NWTB-34,Northwind Traders Beer,10.5,14.00,24 - 12 oz bottles,Beverages
2,NWTB-43,Northwind Traders Coffee,34.5,46.00,16 - 500 g tins,Beverages
3,NWTB-81,Northwind Traders Green Tea,2.0,2.99,20 bags per box,Beverages
4,NWTB-87,Northwind Traders Tea,2.0,4.00,100 count per box,Beverages


,OrderDate,ShipDate
0,2017-01-15,2017-01-22
1,2017-01-20,2017-01-22
2,2017-01-22,2017-01-22
3,2017-01-30,2017-01-31
4,2017-02-06,2017-02-07


,ID,OrderID,ProductCode,Quantity,Price
0,27,30,NWTB-34,100,14.0
1,28,30,NWTDFN-80,30,3.5
2,29,31,NWTDFN-7,10,30.0
3,30,31,NWTDFN-51,10,53.0
4,31,31,NWTDFN-80,10,3.5


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5B: Merge orders with customer information

#### 👇 Tasks 

- ✔️ Merge `df_orders` and `df_customers`.
- ✔️ Store the merged DataFrame to a new variable named `df_5b`.
- ✔️ The resulting DataFrame should have **one row per order**, regardless of whether there's a matching row in `df_customers`.
- ✔️ `df_5b` should **only** contain the following 6 columns in the **same order**:
    - `OrderID`, `OrderDate`, `ShipDate`, `CustomerID`, `CustomerName`, `State`
- ✔️ Store the number of rows in `df_5b` to a new variable named `num_rows_5b`.
    - `num_rows_5b` must be an `int` type.

#### 💡 Hint

1. Use Pandas' `merge()` function.
    - **Example**: `pd.merge(left=df_roster, right=df_students, how=..., on=...)`
2. Think about the column(s) you will use to perform the merge.
3. Make sure you use the correct type of join.

In [32]:
# YOUR CODE BEGINS
conn = sqlite3.connect('NWT.db')
order = '''
    SELECT * FROM tblOrders3;
'''

df_orders = pd.read_sql_query(order, con = conn)
conn.close()
df_5b = pd.merge(left=df_customers, right=df_orders, how='right', on='CustomerID')
#df_5b.drop(columns=['PhoneNum','City', 'Country','ZIP', 'Address',  'ContactName', 'ContactJobTitle'], inplace=True)
#df_5b = df_5b.drop_duplicates(subset='CustomerID', keep='first')
#df_5b = df_5b.reindex(columns=['OrderID', 'OrderDate', 'ShipDate', 'CustomerID', 'CustomerName', 'State'])
df_5b = df_5b[['OrderID', 'OrderDate', 'ShipDate', 'CustomerID', 'CustomerName', 'State']]
num_rows_5b = df_5b.shape[0]

# YOUR CODE ENDS
display(df_5b.head(5))

print(f'There are {num_rows_5b} rows in the merged DataFrame.')

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,State
0,30,1/15/2017,1/22/2017,27,Company AA,NV
1,31,1/20/2017,1/22/2017,4,Company D,NY
2,32,1/22/2017,1/22/2017,12,Company L,NV
3,33,1/30/2017,1/31/2017,8,Company H,OR
4,34,2/6/2017,2/7/2017,4,Company D,NY


There are 48 rows in the merged DataFrame.


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5C: Merge order details into `df_5b`

#### 👇 Tasks 

- ✔️ Merge `df_5b` from the previous part and `df_order_details`.
- ✔️ Store the merged DataFrame to a new variable named `df_5c`.
- ✔️ There might be some orders in `df_orders` that do not have matching rows in `df_order_details`.
- ✔️ **Omit those rows by choosing the proper join type.**
- ✔️ `df_5c` should **only** contain the following 9 columns in the **same order**:
    - `OrderID`, `OrderDate`, `ShipDate`, `CustomerID`, `CustomerName`, `State`, `ProductCode`, `Quantity`, `Price`
- ✔️ Store the number of rows in `df_5c` to a new variable named `num_rows_5c`.
    - `num_rows_5c` must be an `int` type.

In [33]:
# YOUR CODE BEGINS
conn = sqlite3.connect('NWT.db')
df_order_details = pd.read_sql_query('SELECT * FROM tblOrderDetails', con=conn)
df_5c = pd.merge(left=df_5b, right=df_order_details, how='left', on='OrderID')
#df_5c.drop(columns=['ID'], inplace=True)
df_5c = df_5c[['OrderID', 'OrderDate', 'ShipDate', 'CustomerID', 'CustomerName', 'State', 'ProductCode', 'Quantity','Price']]
num_rows_5c = df_5c.shape[0]


# YOUR CODE ENDS
display(df_5c.head(5))
print(f'There are {num_rows_5c} rows in the merged DataFrame.')

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,State,ProductCode,Quantity,Price
0,30,1/15/2017,1/22/2017,27,Company AA,NV,NWTB-34,100.0,14.0
1,30,1/15/2017,1/22/2017,27,Company AA,NV,NWTDFN-80,30.0,3.5
2,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-7,10.0,30.0
3,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-51,10.0,53.0
4,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-80,10.0,3.5


There are 66 rows in the merged DataFrame.


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5D: Explain duplicate `OrderID`s in `df_5c`

Run the two code cells below to display the first 5 rows in `df_5b` and `df_5c`.

In [34]:
display(df_5b.sort_values('OrderID').head(5))

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,State
0,30,1/15/2017,1/22/2017,27,Company AA,NV
1,31,1/20/2017,1/22/2017,4,Company D,NY
2,32,1/22/2017,1/22/2017,12,Company L,NV
3,33,1/30/2017,1/31/2017,8,Company H,OR
4,34,2/6/2017,2/7/2017,4,Company D,NY


In [35]:
display(df_5c.sort_values('OrderID').head(5))

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,State,ProductCode,Quantity,Price
0,30,1/15/2017,1/22/2017,27,Company AA,NV,NWTB-34,100.0,14.0
1,30,1/15/2017,1/22/2017,27,Company AA,NV,NWTDFN-80,30.0,3.5
2,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-7,10.0,30.0
3,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-51,10.0,53.0
4,31,1/20/2017,1/22/2017,4,Company D,NY,NWTDFN-80,10.0,3.5


#### 👇 Tasks

- ✔️ `df_5b` only has 1 row for `OrderID`s `30` and `31`, respectively.
    - In fact, `df_5b` does not contain any duplicate `OrderID` values.
- ✔️ However, `df_5c` has 2 rows for `OrderID` `30` (first two rows) and 3 rows for `OrderID` `31` (next three rows).
- ✔️ <span style="color: blue;">In a few sentences</span>, explain why there are duplicate `OrderID`s in `df_5c`.

<h4 style="color: tomato;">✏️ Your Response</h4>
<p style="color: navy;">
Column duplication usually occurs when the two or data frames have columns with the same name and when the columns are not used in the JOIN statement.

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5E: Merge `df_products` into `df_5c`

#### 👇 Tasks

- ✔️ Using an **inner join**, merge `df_products` into `df_5c`.
- ✔️ Store the merged DataFrame to a new variable named `df_merged`.
- ✔️ `df_merged` should **only** contain the following 10 columns in the **same order**:
    - `OrderID`, `OrderDate`, `ShipDate`, `CustomerID`, `CustomerName`, `ProductCode`, `ProductName`, `Quantity`, `Price`, `StandardCost`

In [36]:
# YOUR CODE BEGINS
df_merged = pd.merge(left=df_products, right=df_5c, how='inner', on='ProductCode')

df_merged = df_merged[['OrderID', 'OrderDate', 'ShipDate', 'CustomerID', 'CustomerName', 'ProductCode','ProductName', 'Quantity','Price', 'StandardCost']]

# YOUR CODE ENDS

display(df_merged.head(5))
print(f'There are {df_merged.shape[0]} rows in the merged DataFrame.')

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,ProductCode,ProductName,Quantity,Price,StandardCost
0,32,1/22/2017,1/22/2017,12,Company L,NWTB-1,Northwind Traders Chai,15.0,18.0,13.5
1,44,3/24/2017,4/1/2017,1,Company A,NWTB-1,Northwind Traders Chai,25.0,18.0,13.5
2,30,1/15/2017,1/22/2017,27,Company AA,NWTB-34,Northwind Traders Beer,100.0,14.0,10.5
3,47,4/8/2017,4/8/2017,6,Company F,NWTB-34,Northwind Traders Beer,300.0,14.0,10.5
4,55,4/5/2017,4/5/2017,29,Company CC,NWTB-34,Northwind Traders Beer,87.0,14.0,10.5


There are 58 rows in the merged DataFrame.


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 5F: Preparing for further analysis 

#### 👇 Tasks

- ✔️ Copy `df_merged` to a new variable named `df_summary` using `copy()`.
    - **Example**: `df_copied = df_original.copy()`
- ✔️ In `df_summary`, rename the `Price` column to `UnitSalePrice`.
- ✔️ In `df_summary`, rename the `StandardCost` column to `UnitCost`.
- ✔️ Filter `df_summary` so that it only contains transactions occured in 2017.
    - Use `ShipDate` to extract the years.
    - You may not see a change in the number of rows after filtering by the year 2017.
        - If that's the case, you're fine! **This means that all orders in your database are 2017 transactions.**

In [76]:
# YOUR CODE BEGINS
df_summary = df_merged.copy()
df_summary.rename(columns = {'Price':'UnitSalePrice', 'StandardCost':'UnitCost'}, inplace = True)
df_summary[df_summary['ShipDate']=='2017']

# YOUR CODE ENDS

display(df_summary.head(3))
print(f'There are {df_summary.shape[0]} rows in the df_summary.')

,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,ProductCode,ProductName,Quantity,UnitSalePrice,UnitCost
0,32,1/22/2017,1/22/2017,12,Company L,NWTB-1,Northwind Traders Chai,15.0,18.0,13.5
1,44,3/24/2017,4/1/2017,1,Company A,NWTB-1,Northwind Traders Chai,25.0,18.0,13.5
2,30,1/15/2017,1/22/2017,27,Company AA,NWTB-34,Northwind Traders Beer,100.0,14.0,10.5


There are 58 rows in the df_summary.


<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Part 6: Compute Accounting Line Items

- The final part of the case will involve computing accounting numbers. 💵💵💵
- We will leave most of these parts up to you with minimal guidance.
- However, your output format **MUST** match the sample formats provided.
- You are free to add new columns to `df_summary` for this part.

<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 6A: Compute company's total revenue in 2017

#### 👇 Tasks 

- ✔️ Using `df_summary`, calculate the company's total revenue from orders.
- ✔️ Store the result in a new variable named `total_revenue`.
    - `total_revenue` should be a `float` type.

In [38]:
total_revenue = df_summary.copy()
total_revenue=total_revenue["Quantity"] * df_summary["UnitSalePrice"]
print(f"The total revenue is {total_revenue.sum()}")


The total revenue is 68137.0


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 6B: Compute revenue in each quarter of 2017

#### 👇 Tasks 

- ✔️ Using `df_summary`, compute revenue in each quarter of 2017.
- ✔️ Assume revenue is recognized on the shipment date (`ShipDate`).
- ✔️ Your output must be a DataFrame named `df_quarterly_revenue` with the following two columns in the same order:
    - `Quarter`: Quarter in integers (e.g., `1`, `2`, `3`, `4`)
    - `Revenue`: Revenue for each quarter
- ✔️ Both `Quarter` and `Revenue` columns should not be used as an index column.
    - `print(df_quarterly_revenue.columns.to_list())` should print out `['Quarter', 'Revenue']`.
- ✔️ There may be one or more quarter without any sales. This means that your DataFrame could only have 3 or less quarters.
- ✔️ Sort `df_quarterly_revenue` by `Quarter` in ascending order.

<div style="display: block; text-align: center; background-color: #EAFAF1; color: #28B463; font-size: 16px; padding: 10px 15px; border: 1px solid #ABEBC6; margin-top: 12px; ">
Sample Output 🥑
</div>

<span style="color: #28B463;">You will likely see different quarters and revenue amounts. The table below is only meant to show the output format.</span>

|   | Quarter | Revenue |
|--:|--------:|--------:|
| 0 |       2 | 54117.5 |
| 1 |       3 | 37116.0 |
| 2 |       4 | 15331.5 |

In [51]:
df_summary['Revenue']= df_summary['Quantity']*df_summary['UnitSalePrice']
df_summary['ShipDate']= pd.to_datetime(df_summary['ShipDate'])
df_summary['Quarter']=df_summary['ShipDate'].dt.quarter
df_summary_info = df_summary[df_summary['ShipDate'].dt.year==2017]
df_quarterly_revenue=df_summary_info.groupby('Quarter')['Revenue'].sum().reset_index()
df_quarterly_revenue.columns = ['Quarter', 'Revenue']
df_quarterly_revenue = df_quarterly_revenue.sort_values(by='Quarter')

#data frame 
display(df_quarterly_revenue.head(5))


,Quarter,Revenue
0,1,22430.5
1,2,42836.5
2,3,2870.0


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 6C: Compute revenue and profit by product in 2017

#### 👇 Tasks

- ✔️ Assume the company had no variances, i.e. the actual cost equaled the `UnitCost` for all products.
- ✔️ Using `df_summary`, compute revenue and profit of every product in 2017.
- ✔️ Your output must be a DataFrame named `df_product_summary` with the following four columns in the same order:
    - `ProductCode`: Unique identifier for each product
    - `ProductName`: Product name
    - `Revenue`: Total revenue of the product
    - `Profit`: Total profit of the product
- ✔️ All 4 columns listed above should not be used as an index column.
    - `print(df_product_summary.columns.to_list())` should print out `['ProductCode', 'ProductName', 'Revenue', 'Profit']`.
- ✔️ Sort `df_product_summary` by `Profit` in descending order.
- ✔️ Show the first 5 rows of `df_product_summary`.

<div style="display: block; text-align: center; background-color: #EAFAF1; color: #28B463; font-size: 16px; padding: 10px 15px; border: 1px solid #ABEBC6; margin-top: 12px; ">
Sample Output 🥑
</div>

<span style="color: #28B463;">You will see different values. The table below is only meant to show the output format.</span>

|    | ProductCode |            ProductName | Revenue | Profit |
|---:|------------:|-----------------------:|--------:|-------:|
|  4 |    ILLINI-2 | Illini Sweatshirt Navy | 15013.0 | 3987.0 |
| 12 |     ARIZ-11 | Wildcat Basic Notebook |  7450.0 | 2350.0 |
|  5 |   ILLINO-10 |     Illini Long Sleeve |  7025.0 | 2800.3 |
|  2 |     YALE-15 |     Handsome Dan Plush |  3000.0 |  750.0 |
| 17 |     ILLSO-7 |  Illini Alumni Sticker |  2455.0 |  678.0 |

In [117]:
df_summary['Cost'] = df_summary['Quantity'] *  df_summary['UnitCost']
df_summary['Revenue']= df_summary['Quantity']*df_summary['UnitSalePrice']
df_summary['Profit'] = df_summary['Revenue'] - df_summary['Cost']
df_product_summary = df_summary[['ProductCode', 'ProductName', 'Revenue','Profit' ]]
df_product_summary= df_product_summary.sort_values(by='Profit',ascending=False)
display(df_product_summary.head(5))


,ProductCode,ProductName,Revenue,Profit
6,NWTB-43,Northwind Traders Coffee,13800.0,3450.0
7,NWTB-43,Northwind Traders Coffee,13800.0,3450.0
44,NWTJP-6,Northwind Traders Boysenberry Spread,3240.0,2490.0
3,NWTB-34,Northwind Traders Beer,4200.0,1050.0
45,NWTJP-6,Northwind Traders Boysenberry Spread,2250.0,562.5


<div style="margin-top: 16px; height: 1px; border-top: 4px dotted black;"></div>

### 📌 Part 6D: Compute revenue and profit by customer in 2017

#### 👇 Tasks

- ✔️ Using `df_summary`, compute revenue and profit of every customer in 2017.
- ✔️ Your output must be a DataFrame named `df_customer_summary` with the following four columns in the same order:
    - `CustomerID`: Unique identifier for each product
    - `CustomerName`: Customer name (not contact name)
    - `Revenue`: Total revenue of the customer
    - `Profit`: Total profit of the customer
- ✔️ All 4 columns listed above should not be used as an index column.
    - `print(df_customer_summary.columns.to_list())` should print out `['CustomerID', 'CustomerName', 'Revenue', 'Profit']`.
- ✔️ Sort `df_customer_summary` by `Profit` in descending order.
- ✔️ Show the first 5 rows of `df_customer_summary`.

<div style="display: block; text-align: center; background-color: #EAFAF1; color: #28B463; font-size: 16px; padding: 10px 15px; border: 1px solid #ABEBC6; margin-top: 12px; ">
Sample Output 🥑
</div>

<span style="color: #28B463;">You will see different values. The table below is only meant to show the output format.</span>

|    | CustomerID | CustomerName | Revenue | Profit |
|---:|-----------:|-------------:|--------:|-------:|
|  9 |          8 |    Company D | 10452.5 | 3351.0 |
|  1 |          4 |    Company A |  8510.0 | 2925.0 |
|  6 |          7 |    Company T |  3535.0 | 2310.5 |
| 12 |          3 |    Company B |  4180.5 | 1949.0 |
|  5 |         10 |   Company CC |  3500.5 | 1100.0 |

In [119]:


df_summary["Revenue"] = df_summary["Quantity"] * df_summary["UnitSalePrice"]
df_summary["Profit"] = df_summary["Revenue"] - (df_summary["UnitCost"] * df_summary["Quantity"])
df_customer_summary =df_summary[['CustomerID', 'CustomerName', 'Revenue', 'Profit']]
df_customer_summary = df_customer_summary.sort_values(by='Profit',ascending=False)
display(df_customer_summary.head(4))



,CustomerID,CustomerName,Revenue,Profit
6,28,Company BB,13800.0,3450.0
7,7,Company G,13800.0,3450.0
44,4,Company D,3240.0,2490.0
3,6,Company F,4200.0,1050.0


<div style="margin-top: 16px; height: 1px; border-top: 10px solid black;"></div>

## Final Step

### Check if all code cells run without an error

If you have completed all the parts, congratulations 🎉! Your final step is to ensure that your notebook runs without an error. Click on the "Kernel" menu and select "Restart Kernel and Run All Cells..." option.

![Restart Kernel and Run All Cells](https://accy570-fa2020-course-site-assets.s3-us-west-2.amazonaws.com/images/jupyter-clear-kernel-run-all-cells-menu.png)

Once your notebook finish running, make sure that the code cell at the bottom prints "Success 🎈!". 

![Success Message](https://accy570-fa2020-course-site-assets.s3-us-west-2.amazonaws.com/images/jupyter-clear-kernel-run-all-cells-menu-result-success.png)

If you don't see any printed output, it means that one or more of your cells contain an error. **Fix all errors until you see the success message.** Failing to do so may result in **significant loss of points** since the autograder will fail to run.

In [44]:
print('Success 🎈!')

Success 🎈!
